# Classification automatique d'articles de presse (AG News)

Le but du projet est de construire un modèle capable de deviner la rubrique d'un article de presse à partir de son texte, puis de le rendre utilisable par n'importe qui à travers une API et une petite interface web.

On travaille sur le dataset AG News (Hugging Face, `fancyzhx/ag_news`) : un peu plus de 127 000 articles en anglais, collectés en 2004-2005, répartis en 4 catégories : World, Sports, Business et Sci/Tech. Pour chaque article on a seulement le titre et le chapeau, pas le texte complet.

Ce notebook regroupe toutes les étapes :
1. exploration des données ;
2. nettoyage et preprocessing ;
3. modèles classiques (TF-IDF + Naive Bayes, régression logistique, SVM, Random Forest) ;
4. modèles de deep learning (LSTM et DistilBERT) ;
5. comparaison de tous les modèles et choix de celui qu'on déploie dans l'API.

Il est prévu pour Google Colab avec un GPU (Exécution > Modifier le type d'exécution > GPU T4), surtout pour DistilBERT. L'exécution complète prend environ une demi-heure.

## 0. Installation et imports

`datasets` sert à télécharger AG News depuis Hugging Face, `nltk` fournit la liste de stopwords et le lemmatiseur, `wordcloud` les nuages de mots et `transformers` le modèle DistilBERT. Sur Colab la plupart sont déjà installés, la commande ne fait alors rien.

In [ ]:
%pip install -q datasets nltk wordcloud transformers accelerate

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

## 1. Chargement des données

Le dataset est déjà séparé en un jeu d'entraînement et un jeu de test. Les catégories sont codées de 0 à 3, leurs noms sont rangés dans les métadonnées. On passe tout en DataFrame pandas et on ajoute une colonne `label_text` avec le nom de la catégorie, plus pratique à lire.

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
dataset

In [ ]:
# Noms des catégories stockés dans les métadonnées du dataset
noms_categories = dataset["train"].features["label"].names
noms_categories

In [ ]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

for df in (train_df, test_df):
    df["label_text"] = df["label"].map(dict(enumerate(noms_categories)))

print("Train :", train_df.shape)
print("Test  :", test_df.shape)
print("Total :", len(train_df) + len(test_df), "articles")

In [ ]:
train_df.head()

On a 127 600 articles : 120 000 pour l'entraînement et 7 600 pour le test, ce qui est très largement suffisant pour entraîner des modèles sans trop de risque de surapprentissage.

Chaque texte est le titre suivi directement du chapeau, sans séparateur. On voit tout de suite que les textes ne sont pas propres : la source apparaît au début (« (Reuters) Reuters - ») et il y a des `\` au milieu des phrases.

## 2. Exploration

### 2.1 Structure, valeurs manquantes

In [ ]:
train_df.info()

In [ ]:
for nom, df in (("train", train_df), ("test", test_df)):
    print(f"{nom} : valeurs manquantes = {df.isnull().sum().sum()}, textes vides = {(df['text'].str.strip() == '').sum()}")

Trois colonnes avec les bons types, aucune valeur manquante et aucun texte vide. Rien à corriger de ce côté.

### 2.2 Doublons

Un doublon exact, c'est le même texte au caractère près. Mais on s'est vite rendu compte que beaucoup d'articles reviennent avec juste une ponctuation ou des espaces différents. Pour les repérer, on fabrique une clé qui ne garde que les lettres en minuscules, et on compare les clés.

In [ ]:
print("Doublons exacts — train :", train_df.duplicated(subset="text").sum())
print("Doublons exacts — test  :", test_df.duplicated(subset="text").sum())
print("Articles identiques entre train et test :", len(set(train_df["text"]) & set(test_df["text"])))

In [ ]:
# Quasi-doublons : on compare les textes en ne gardant que les lettres en minuscules
def normaliser(texte):
    return re.sub(r"[^a-z]", "", texte.lower())

train_df["cle"] = train_df["text"].apply(normaliser)
test_df["cle"] = test_df["text"].apply(normaliser)

print("Quasi-doublons — train :", train_df.duplicated(subset="cle").sum())
print("Quasi-doublons — test  :", test_df.duplicated(subset="cle").sum())
print("Quasi-doublons entre train et test :", len(set(train_df["cle"]) & set(test_df["cle"])))

In [ ]:
# Un même article classé dans plusieurs catégories différentes ?
groupes = train_df[train_df.duplicated(subset="cle", keep=False)].groupby("cle")["label_text"]
contradictoires = groupes.nunique()
print("Articles en double avec des catégories différentes :", (contradictoires > 1).sum())

exemple = contradictoires[contradictoires > 1].index[0]
train_df.loc[train_df["cle"] == exemple, ["text", "label_text"]]

Il n'y a aucun doublon exact, mais 424 quasi-doublons dans le train, et surtout 38 articles qu'on retrouve à la fois dans le train et dans le test. C'est une fuite de données : le modèle serait évalué sur des articles qu'il a déjà vus pendant l'entraînement, et son score serait un peu gonflé.

On trouve aussi 44 articles présents plusieurs fois avec des catégories différentes. Dans l'exemple, le rachat de Joyo.com par Amazon est classé une fois en Business et une fois en Sci/Tech. On ne peut pas savoir quelle étiquette est la bonne, donc on supprimera ces articles.

### 2.3 Répartition des catégories

In [ ]:
repartition = pd.DataFrame({
    "train": train_df["label_text"].value_counts(),
    "test": test_df["label_text"].value_counts(),
})
repartition["train_%"] = (repartition["train"] / repartition["train"].sum() * 100).round(1)
repartition["test_%"] = (repartition["test"] / repartition["test"].sum() * 100).round(1)
repartition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ordre = noms_categories

sns.countplot(data=train_df, x="label_text", order=ordre, ax=axes[0], color="steelblue")
axes[0].set_title("Répartition des catégories — train")
axes[0].set_xlabel("Catégorie")
axes[0].set_ylabel("Nombre d'articles")

sns.countplot(data=test_df, x="label_text", order=ordre, ax=axes[1], color="darkorange")
axes[1].set_title("Répartition des catégories — test")
axes[1].set_xlabel("Catégorie")
axes[1].set_ylabel("Nombre d'articles")

plt.tight_layout()
plt.show()

Le dataset est parfaitement équilibré : 30 000 articles par catégorie dans le train et 1 900 dans le test. Pas besoin de rééquilibrer, et l'accuracy sera une mesure fiable. On suivra quand même le F1 macro, qui donne le même poids à chaque catégorie.

### 2.4 Longueur des articles

In [ ]:
for df in (train_df, test_df):
    df["nb_caracteres"] = df["text"].str.len()
    df["nb_mots"] = df["text"].str.split().str.len()

train_df[["nb_caracteres", "nb_mots"]].describe().round(1)

In [ ]:
train_df.groupby("label_text")["nb_mots"].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(train_df["nb_mots"], bins=60, ax=axes[0], color="steelblue")
axes[0].axvline(train_df["nb_mots"].median(), color="red", linestyle="--", label="médiane")
axes[0].set_title("Distribution du nombre de mots par article (train)")
axes[0].set_xlabel("Nombre de mots")
axes[0].set_ylabel("Nombre d'articles")
axes[0].legend()

sns.boxplot(data=train_df, x="label_text", y="nb_mots", order=ordre, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Nombre de mots par catégorie (train)")
axes[1].set_xlabel("Catégorie")
axes[1].set_ylabel("Nombre de mots")

plt.tight_layout()
plt.show()

Les textes sont courts : 37 mots en médiane, 177 au maximum. C'est logique puisqu'on n'a que le titre et le chapeau. Les quatre catégories ont à peu près la même longueur (entre 36 et 39 mots en médiane), donc la longueur ne permet pas de les distinguer, c'est le vocabulaire qui fera la différence. Ces textes courts nous arrangent pour DistilBERT : 128 tokens suffiront.

### 2.5 Quelques exemples par catégorie

In [ ]:
for categorie in ordre:
    print(f"=== {categorie.upper()} ===")
    for texte in train_df.loc[train_df["label_text"] == categorie, "text"].iloc[:2]:
        print("-", texte[:250])
    print()

Le vocabulaire est assez typé : referendum, president, troops pour World, Olympic, homered pour Sports, Wall Street, stock market pour Business. On retrouve aussi presque partout la mention de la source au début du texte.

### 2.6 Bruit dans les textes

Les articles ont été récupérés automatiquement sur des sites web et il reste des morceaux de code HTML. On compte combien d'articles sont touchés par chaque type de bruit.

In [ ]:
motifs = {
    "entité #39; (apostrophe)": r"#39;",
    "entité #36; (dollar)": r"#36;",
    "entité quot; (guillemet)": r"quot;",
    "balise HTML (&lt;...&gt;)": r"&lt;",
    "barre oblique inverse \\": r"\\",
    "lien http": r"http",
    "mention de source (Reuters) Reuters -": r"\(([^()]{1,40})\)\s*\1\s*-",
}

bruit = pd.DataFrame({
    "nb_articles": {nom: train_df["text"].str.contains(m, regex=True).sum() for nom, m in motifs.items()}
})
bruit["%_du_train"] = (bruit["nb_articles"] / len(train_df) * 100).round(2)
bruit.sort_values("nb_articles", ascending=False)

In [ ]:
# Exemple d'article contenant de la publicité HTML
print(train_df.loc[train_df["text"].str.contains("ADVERTISEMENT"), "text"].iloc[0][:600])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
b = bruit.sort_values("nb_articles")
ax.barh(b.index, b["nb_articles"], color="indianred")
ax.set_title("Nombre d'articles du train contenant chaque type de bruit")
ax.set_xlabel("Nombre d'articles")
plt.tight_layout()
plt.show()

Un quart des articles (24,8 %) contient `#39;`, un code HTML mal converti qui remplace l'apostrophe (« Google #39;s » au lieu de « Google's »). `#36;` remplace le dollar et `quot;` les guillemets. 13,4 % des articles commencent par une mention de source, 11 % contiennent des `\`, et 4,4 % ont des balises HTML, parfois avec un bloc de publicité complet (ici une pub IBM au milieu d'un article). Tout ça n'a aucun rapport avec le sujet de l'article, il faudra l'enlever.

### 2.7 Sources des articles

On extrait la source citée au début de l'article (Reuters, AP, AFP...) et on regarde dans quelles catégories tombent ses articles.

In [ ]:
train_df["source"] = train_df["text"].str.extract(r"\(([^()]{1,40})\)\s*\1\s*-")[0]
top_sources = train_df["source"].value_counts().head(8).index

tableau = pd.crosstab(train_df.loc[train_df["source"].isin(top_sources), "source"],
                      train_df.loc[train_df["source"].isin(top_sources), "label_text"])
tableau

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
part = tableau.div(tableau.sum(axis=1), axis=0) * 100
sns.heatmap(part, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% des articles de la source"}, ax=ax)
ax.set_title("Répartition des articles de chaque source par catégorie (%)")
ax.set_xlabel("Catégorie")
ax.set_ylabel("Source")
plt.tight_layout()
plt.show()

C'est le résultat le plus intéressant de l'exploration. Certaines sources ne publient que dans une seule catégorie : Canadian Press est à 100 % en World, PC World et SPACE.com à 100 % en Sci/Tech, AFP à 77 % en World. Un modèle pourrait donc apprendre à reconnaître la source plutôt que le sujet. Ça marcherait sur ce dataset, mais pas dans notre application, où l'utilisateur colle un texte sans mention de source. On va donc supprimer ces mentions pour forcer le modèle à apprendre sur le contenu.

Ce qu'on retient de l'exploration :

| Constat | Ce qu'on fait |
|---|---|
| 4 catégories parfaitement équilibrées | rien |
| textes courts (37 mots) | 128 tokens max pour DistilBERT |
| codes HTML, `\`, liens, publicités | fonction de nettoyage |
| sources liées à certaines catégories | suppression des mentions de source |
| quasi-doublons, étiquettes contradictoires, articles communs train/test | suppression |

## 3. Nettoyage et preprocessing

### 3.1 Suppression du bruit

La fonction `supprimer_bruit` corrige les codes HTML (`#39;` devient une apostrophe, `#36;` un dollar, `quot;` un guillemet), enlève les publicités, les balises, les liens, les liens boursiers de Reuters et les mentions de source, sous leurs deux formes : « (Reuters) Reuters - » et « NEW YORK (Reuters) - ».

On garde les majuscules et la ponctuation à cette étape, parce que DistilBERT a besoin de vraies phrases.

In [ ]:
import re
import html
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

In [ ]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True);

In [ ]:
def supprimer_bruit(texte):
    texte = re.sub(r" ?&?#(\d+);", lambda m: chr(int(m.group(1))), texte) # " #39;" -> '   " #36;" -> $
    texte = re.sub(r"&?quot;", '"', texte)                                # quot; -> "
    texte = html.unescape(texte)                                         # &lt; -> <   &gt; -> >
    texte = re.sub(r"<p>\s*ADVERTISEMENT.*$", " ", texte)                # bloc publicitaire en fin d'article
    texte = re.sub(r"<[^>]*>", " ", texte)                               # balises HTML
    texte = re.sub(r"https?://\S+|www\.\S+", " ", texte)                 # liens
    texte = re.sub(r"\([^()]{0,15}Quote, Profile, Research\)", " ", texte) # lien boursier Reuters
    texte = re.sub(r"\(([^()]{1,40})\)\s*\1\s*-\s*", " ", texte)          # "(Reuters) Reuters - "
    texte = re.sub(r"\([A-Za-z][A-Za-z. ]{1,30}\)\s+-\s+", " ", texte)    # "NEW YORK (Reuters) - "
    texte = texte.replace("\\", " ")                                     # barres obliques inverses
    texte = re.sub(r"\s+", " ", texte)                                   # espaces multiples
    return texte.strip()

In [ ]:
exemples = [
    train_df["text"].iloc[0],
    train_df.loc[train_df["text"].str.contains("#39;"), "text"].iloc[0],
    train_df.loc[train_df["text"].str.contains("ADVERTISEMENT"), "text"].iloc[0],
]
for texte in exemples:
    print("AVANT :", texte[:300])
    print("APRÈS :", supprimer_bruit(texte)[:300])
    print()

Sur les trois exemples, la mention de source a disparu, « Web #39;s » est redevenu « Web's » et le bloc de pub IBM a été supprimé en entier. Le nom de la ville (NEW YORK) reste, ce n'est pas gênant.

On applique la fonction à tout le dataset et on vérifie qu'il ne reste plus de bruit :

In [ ]:
train_df["text"] = train_df["text"].apply(supprimer_bruit)
test_df["text"] = test_df["text"].apply(supprimer_bruit)

motifs = {"#39;": r"#39;", "quot;": r"quot;", "&lt;": r"&lt;", "<balise>": r"<[^>]+>",
          "http": r"http", "ADVERTISEMENT": r"ADVERTISEMENT", "\\": r"\\",
          "Quote, Profile, Research": r"Quote, Profile, Research",
          "(Source) Source -": r"\(([^()]{1,40})\)\s*\1\s*-", "VILLE (Source) -": r"\([A-Za-z][A-Za-z. ]{1,30}\)\s+-\s+"}
for nom, m in motifs.items():
    print(f"{nom:<20} restants dans le train : {train_df['text'].str.contains(m, regex=True).sum()}")

Il ne reste presque rien : 4 articles avec le mot ADVERTISEMENT (dont un où il est employé normalement) et 2 liens boursiers écrits autrement. Six cas sur 120 000, on les laisse.

### 3.2 Doublons, étiquettes contradictoires et fuite de données

On recalcule la clé après le nettoyage : deux articles qui ne différaient que par la mention de source deviennent identiques, on va donc en trouver plus que pendant l'exploration.

D'abord, les articles qui ont plusieurs catégories : on supprime toutes leurs occurrences.

In [ ]:
def normaliser(texte):
    return re.sub(r"[^a-z]", "", texte.lower())

train_df["cle"] = train_df["text"].apply(normaliser)
test_df["cle"] = test_df["text"].apply(normaliser)

# Clés associées à plusieurs catégories (dans le train, le test ou entre les deux)
tout = pd.concat([train_df[["cle", "label"]], test_df[["cle", "label"]]])
cles_contradictoires = set(tout.groupby("cle")["label"].nunique().loc[lambda s: s > 1].index)

avant_train, avant_test = len(train_df), len(test_df)
train_df = train_df[~train_df["cle"].isin(cles_contradictoires)]
test_df = test_df[~test_df["cle"].isin(cles_contradictoires)]

print("Articles distincts aux catégories contradictoires :", len(cles_contradictoires))
print(f"Lignes supprimées — train : {avant_train - len(train_df)} | test : {avant_test - len(test_df)}")

170 articles différents ont des étiquettes contradictoires, ce qui fait 331 lignes en moins dans le train et 16 dans le test.

Ensuite on ne garde qu'un exemplaire de chaque article :

In [ ]:
avant_train, avant_test = len(train_df), len(test_df)

train_df = train_df.drop_duplicates(subset="cle").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="cle").reset_index(drop=True)

print(f"Train : {avant_train} -> {len(train_df)} ({avant_train - len(train_df)} doublons supprimés)")
print(f"Test  : {avant_test} -> {len(test_df)} ({avant_test - len(test_df)} doublons supprimés)")

1 335 doublons en moins dans le train. Enfin, on retire du test les articles déjà présents dans le train :

In [ ]:
communs = test_df["cle"].isin(train_df["cle"])
print("Articles du test déjà présents dans le train :", communs.sum())

test_df = test_df[~communs].reset_index(drop=True)
print("Taille du test après suppression :", len(test_df))
print("Articles communs restants :", len(set(train_df["cle"]) & set(test_df["cle"])))

156 articles du test étaient déjà dans le train. On les enlève du test (et pas du train, pour ne pas perdre de données d'apprentissage). Il ne reste plus aucun article commun.

In [ ]:
repartition = pd.DataFrame({
    "train": train_df["label_text"].value_counts(),
    "test": test_df["label_text"].value_counts(),
})
repartition["total"] = repartition["train"] + repartition["test"]
repartition["total_%"] = (repartition["total"] / repartition["total"].sum() * 100).round(2)
repartition.loc["TOTAL"] = repartition[["train", "test", "total"]].sum().tolist() + [100.0]

print(f"Dataset final : {len(train_df) + len(test_df):,} articles (sur 127 600 au départ)")
repartition

Au final on garde 125 759 articles (118 334 en train, 7 425 en test). On n'a supprimé que 1,4 % des données et les catégories restent équilibrées, entre 24,6 % et 25,2 %.

### 3.3 Preprocessing du texte

Pour les modèles classiques et le LSTM, on simplifie encore le texte avec la fonction `preprocesser` : passage en minuscules, suppression des chiffres et de la ponctuation, suppression des stopwords (les mots outils comme the, and, of) et lemmatisation, qui ramène chaque mot à sa forme de base (companies devient company).

On filtre les stopwords deux fois : avant la lemmatisation, parce que le lemmatiseur abîme certains mots outils (was devient wa, has devient ha), et après, pour ceux qui ne deviennent des stopwords qu'une fois lemmatisés. Le `lru_cache` évite de lemmatiser plusieurs fois le même mot, ce qui accélère beaucoup le traitement.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

@lru_cache(maxsize=None)
def lemmatiser(mot):
    return lemmatizer.lemmatize(mot)

def preprocesser(texte):
    texte = texte.lower()                                      # minuscules
    texte = re.sub(r"[^a-z\s]", " ", texte)                    # garde uniquement les lettres
    mots = [m for m in texte.split() if m not in stop_words and len(m) > 2]
    mots = [lemmatiser(m) for m in mots]                       # lemmatisation
    mots = [m for m in mots if m not in stop_words]            # 2e filtre après lemmatisation
    return " ".join(mots)

print("Nombre de stopwords NLTK :", len(stop_words))
for mot in ["companies", "markets", "prices", "games", "players", "computers", "elections"]:
    print(f"  {mot:<10} -> {lemmatiser(mot)}")

In [ ]:
%%time
train_df["text_clean"] = train_df["text"].apply(preprocesser)
test_df["text_clean"] = test_df["text"].apply(preprocesser)

In [ ]:
for i in range(3):
    print("ORIGINAL :", train_df.loc[i, "text"])
    print("NETTOYÉ  :", train_df.loc[i, "text_clean"])
    print()

Grâce au cache, tout le dataset passe en moins d'une seconde. Le texte nettoyé ne garde que les mots qui ont du sens. On garde les deux colonnes : `text` (sans bruit mais avec les phrases complètes) servira à DistilBERT, `text_clean` aux autres modèles.

### 3.4 Vocabulaire de chaque catégorie

In [ ]:
categories = noms_categories

def top_mots(df, categorie, n):
    mots = " ".join(df.loc[df["label_text"] == categorie, "text_clean"]).split()
    return Counter(mots).most_common(n)

def afficher_top_mots(df, titre, n=15):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for ax, categorie in zip(axes, categories):
        top = pd.DataFrame(top_mots(df, categorie, n), columns=["mot", "frequence"])
        sns.barplot(data=top, y="mot", x="frequence", ax=ax, color="steelblue")
        ax.set_title(categorie)
        ax.set_xlabel("Fréquence")
        ax.set_ylabel("")
    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()

afficher_top_mots(train_df, "15 mots les plus fréquents par catégorie (train)")

Chaque catégorie a ses mots à elle : iraq, president, killed pour World, game, win, team pour Sports, oil, company, stock pour Business, microsoft, software, internet pour Sci/Tech. Mais quelques mots comme new et year sont en haut de toutes les catégories, ils n'aident pas à les séparer.

Plutôt que de choisir ces mots à la main, on prend ceux qui sont dans le top 30 des quatre catégories à la fois, et on les ajoute aux stopwords :

In [ ]:
tops = [{mot for mot, _ in top_mots(train_df, c, 30)} for c in categories]
stopwords_corpus = set.intersection(*tops)
print("Mots communs au top 30 des 4 catégories :", sorted(stopwords_corpus))

In [ ]:
stop_words = stop_words | stopwords_corpus

train_df["text_clean"] = train_df["text"].apply(preprocesser)
test_df["text_clean"] = test_df["text"].apply(preprocesser)

afficher_top_mots(train_df, "15 mots les plus fréquents par catégorie (après retrait des stopwords du corpus)")

La règle sort seulement new et year. Une fois retirés, d'autres mots plus utiles remontent (microsoft passe en tête de Sci/Tech, quarter et billion apparaissent en Business). « said » reste fréquent mais il n'est pas dans le top des quatre catégories (pas en Sports), on le garde ; le TF-IDF réduira de toute façon son poids.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, categorie in zip(axes.flatten(), categories):
    texte = " ".join(train_df.loc[train_df["label_text"] == categorie, "text_clean"])
    nuage = WordCloud(width=700, height=400, background_color="white", max_words=80,
                      colormap="viridis", random_state=42).generate(texte)
    ax.imshow(nuage, interpolation="bilinear")
    ax.set_title(categorie, fontsize=14)
    ax.axis("off")
fig.suptitle("Nuages de mots par catégorie (train)", fontsize=16)
plt.tight_layout()
plt.show()

Les nuages de mots sont très différents d'une catégorie à l'autre, ce qui est bon signe pour la suite. On voit bien l'époque : guerre en Irak, élection américaine (john kerry), prix du pétrole. Par contre company et microsoft apparaissent à la fois en Business et en Sci/Tech : on s'attend à ce que ce soit là que les modèles se trompent le plus.

In [ ]:
vides_train = (train_df["text_clean"].str.strip() == "").sum()
vides_test = (test_df["text_clean"].str.strip() == "").sum()
print("Textes vides après preprocessing — train :", vides_train, "| test :", vides_test)

train_df = train_df[train_df["text_clean"].str.strip() != ""].reset_index(drop=True)
test_df = test_df[test_df["text_clean"].str.strip() != ""].reset_index(drop=True)

In [ ]:
brut = dataset["train"].to_pandas()["text"]

nb_mots_brut = brut.str.split().str.len()
nb_mots_clean = train_df["text_clean"].str.split().str.len()

vocab_brut = len(set(" ".join(brut).split()))
vocab_clean = len(set(" ".join(train_df["text_clean"]).split()))

print(f"Mots par article (médiane) : {nb_mots_brut.median():.0f} -> {nb_mots_clean.median():.0f}")
print(f"Total de mots              : {nb_mots_brut.sum():,} -> {nb_mots_clean.sum():,}")
print(f"Taille du vocabulaire      : {vocab_brut:,} -> {vocab_clean:,} mots distincts")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.arange(0, 120, 2)
sns.histplot(nb_mots_brut, bins=bins, color="lightgray", label="texte brut", ax=ax)
sns.histplot(nb_mots_clean, bins=bins, color="steelblue", label="après preprocessing", ax=ax)
ax.set_title("Nombre de mots par article avant et après preprocessing (train)")
ax.set_xlabel("Nombre de mots")
ax.set_ylabel("Nombre d'articles")
ax.legend()
plt.tight_layout()
plt.show()

Aucun texte n'est devenu vide après le preprocessing. Un article passe de 37 à 24 mots en médiane, et le vocabulaire passe de 188 110 à 54 450 mots différents (−71 %). Dans le texte brut, « Google », « google » et « Google's » comptaient comme trois mots, sans parler des chiffres et du bruit HTML. Les données sont prêtes.

## 4. Modèles classiques

### 4.1 Vectorisation TF-IDF

Un modèle ne lit pas du texte, il faut le transformer en nombres. Le TF-IDF donne à chaque mot d'un article un poids qui est d'autant plus fort que le mot est fréquent dans cet article et rare dans le reste du corpus. Un mot comme touchdown pèse lourd, un mot qu'on trouve partout pèse peu.

Quelques réglages : on garde les mots seuls et les paires de mots (`ngram_range=(1, 2)`, pour avoir wall street ou prime minister), on ignore les termes présents dans moins de 3 articles, on limite le vocabulaire à 100 000 termes et on utilise `sublinear_tf` pour qu'un mot répété dix fois ne compte pas dix fois plus.

Le vectoriseur apprend son vocabulaire sur le train seulement. Le test doit rester des textes jamais vus, comme ceux que tapera un utilisateur.

In [ ]:
import io
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

In [ ]:
train_df[["text_clean", "label_text"]].head()

In [ ]:
X_train_texte, y_train = train_df["text_clean"], train_df["label_text"]
X_test_texte, y_test = test_df["text_clean"], test_df["label_text"]
categories = sorted(y_train.unique())
categories

In [ ]:
vectoriseur = TfidfVectorizer(
    ngram_range=(1, 2),     # mots seuls + paires de mots consécutifs
    min_df=3,               # ignore les termes présents dans moins de 3 articles
    max_features=100_000,   # garde les 100 000 termes les plus fréquents
    sublinear_tf=True,      # 1 + log(tf) : atténue les mots répétés
)

debut = time.time()
X_train = vectoriseur.fit_transform(X_train_texte)   # apprend le vocabulaire sur le train uniquement
X_test = vectoriseur.transform(X_test_texte)
temps_tfidf = time.time() - debut

print(f"Vectorisation : {temps_tfidf:.1f} s")
print("X_train :", X_train.shape, "| X_test :", X_test.shape)
print(f"Taux de remplissage : {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.3f} %")

In [ ]:
# Exemples de termes du vocabulaire : unigrammes et bigrammes
vocabulaire = vectoriseur.get_feature_names_out()
bigrammes = [t for t in vocabulaire if " " in t]
print("Unigrammes :", len(vocabulaire) - len(bigrammes), "| Bigrammes :", len(bigrammes))
print("Exemples de bigrammes :", bigrammes[::5000][:12])

On obtient une matrice de 118 334 articles sur 100 000 termes, remplie à 0,029 % seulement : chaque article contient une trentaine de termes. scikit-learn stocke ça sous forme de matrice creuse (seulement les valeurs non nulles), sinon il faudrait près de 95 Go de mémoire. Les paires de mots font 77 % du vocabulaire.

### 4.2 Évaluation

Tous les modèles passent par la même fonction, pour que la comparaison soit juste. On mesure l'accuracy, le F1 macro (notre métrique principale), le temps d'entraînement, le temps de prédiction et la taille du modèle sauvegardé. Pour la taille, on compte le vectoriseur en plus du classifieur, parce que l'API aura besoin des deux.

In [ ]:
def taille_mo(objet):
    """Taille de l'objet une fois sauvegardé avec joblib, en Mo."""
    tampon = io.BytesIO()
    joblib.dump(objet, tampon)
    return tampon.getbuffer().nbytes / 1e6


resultats = []
predictions = {}
modeles = {}   # on garde les modèles entraînés pour la partie 6

def evaluer(nom, modele):
    debut = time.time()
    modele.fit(X_train, y_train)
    temps_entrainement = time.time() - debut

    debut = time.time()
    y_pred = modele.predict(X_test)
    temps_prediction_us = (time.time() - debut) / len(y_test) * 1e6   # temps moyen par article, en microsecondes

    predictions[nom] = y_pred
    modeles[nom] = modele
    resultats.append({
        "modele": nom,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "temps_entrainement_s": temps_entrainement,
        "prediction_us_par_article": temps_prediction_us,
        "taille_mo": taille_mo(vectoriseur) + taille_mo(modele),   # le modèle déployé = TF-IDF + classifieur
    })
    r = resultats[-1]
    print(f"{nom} : accuracy = {r['accuracy']:.4f}, F1 macro = {r['f1_macro']:.4f}, "
          f"entraînement = {temps_entrainement:.1f} s, taille = {r['taille_mo']:.1f} Mo")

### 4.3 Réglage des hyperparamètres

Avant d'entraîner les modèles, on choisit leurs réglages par validation croisée sur le train (3 plis), sans jamais toucher au test. Pour Naive Bayes on règle le lissage `alpha`, pour la régression logistique et le SVM le paramètre `C` : plus il est petit, plus le modèle est contraint, ce qui limite le surapprentissage. Le Random Forest est trop long à régler, on garde ses paramètres par défaut.

In [ ]:
grilles = {
    "Naive Bayes": (MultinomialNB(), {"alpha": [0.05, 0.1, 0.5, 1.0]}),
    "Régression logistique": (LogisticRegression(max_iter=1000), {"C": [1, 5, 10]}),
    "SVM linéaire": (LinearSVC(), {"C": [0.1, 0.5, 1]}),
}

meilleurs_params = {}
for nom, (modele, grille) in grilles.items():
    debut = time.time()
    recherche = GridSearchCV(modele, grille, cv=3, scoring="f1_macro", n_jobs=-1)
    recherche.fit(X_train, y_train)
    meilleurs_params[nom] = recherche.best_params_
    scores = dict(zip([str(p) for p in recherche.cv_results_["params"]],
                      recherche.cv_results_["mean_test_score"].round(4)))
    print(f"{nom} ({time.time() - debut:.0f} s) : {scores} → meilleur : {recherche.best_params_}")

Naive Bayes n'est presque pas sensible à `alpha` (tous les scores sont autour de 0,896). Pour la régression logistique et le SVM, c'est la plus forte régularisation qui gagne (C = 1 et C = 0,1) : avec 100 000 termes pour 118 000 articles, un modèle peu contraint finit par apprendre le train par cœur.

### 4.4 Entraînement des quatre modèles

Naive Bayes calcule pour chaque catégorie la probabilité de voir chaque mot, c'est le modèle de référence en classification de texte. La régression logistique apprend un poids par mot et par catégorie et donne directement des probabilités. Le SVM linéaire cherche les frontières qui séparent le mieux les catégories, il marche très bien sur ce genre de données, mais ne donne pas de probabilités. Le Random Forest vote avec 100 arbres de décision.

In [ ]:
evaluer("Naive Bayes", MultinomialNB(**meilleurs_params["Naive Bayes"]))

In [ ]:
evaluer("Régression logistique", LogisticRegression(max_iter=1000, **meilleurs_params["Régression logistique"]))

In [ ]:
evaluer("SVM linéaire", LinearSVC(**meilleurs_params["SVM linéaire"]))

In [ ]:
evaluer("Random Forest", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))

### 4.5 Comparaison

In [ ]:
comparaison = pd.DataFrame(resultats).set_index("modele").sort_values("f1_macro", ascending=False)
comparaison.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
comparaison[["accuracy", "f1_macro"]].plot.bar(ax=ax, color=["steelblue", "darkorange"], rot=0)
ax.set_ylim(0.8, 0.95)
ax.set_title("Accuracy et F1 macro sur le jeu de test")
ax.set_xlabel("")
ax.set_ylabel("Score")
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

comparaison["temps_entrainement_s"].plot.bar(ax=axes[0], color="seagreen", rot=0, logy=True)
axes[0].set_title("Temps d'entraînement (échelle logarithmique)")
axes[0].set_ylabel("Secondes")
axes[0].bar_label(axes[0].containers[0], fmt="%.1f s")

comparaison["taille_mo"].plot.bar(ax=axes[1], color="indianred", rot=0, logy=True)
axes[1].set_title("Taille du modèle déployé : TF-IDF + classifieur (échelle log.)")
axes[1].set_ylabel("Mo")
axes[1].bar_label(axes[1].containers[0], fmt="%.1f Mo")

for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

Le SVM linéaire est le meilleur (F1 = 0,911), juste devant la régression logistique (0,909) et Naive Bayes (0,902). L'écart entre les trois est faible : 0,2 point entre le SVM et la régression logistique, ça fait une quinzaine d'articles sur 7 425. Attention, l'axe du premier graphique commence à 0,80, ce qui grossit visuellement les différences.

Le Random Forest est le moins bon (0,890), le plus long à entraîner et surtout il pèse 612 Mo, contre 7,4 Mo pour le SVM et la régression logistique. Ses arbres sont développés jusqu'au bout et stockent des milliers de nœuds. Il est éliminé.

### 4.6 Où les modèles se trompent

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, nom in zip(axes.ravel(), comparaison.index):
    matrice = confusion_matrix(y_test, predictions[nom], labels=categories, normalize="true") * 100
    sns.heatmap(matrice, annot=True, fmt=".1f", cmap="Blues", cbar=False, ax=ax,
                xticklabels=categories, yticklabels=categories)
    ax.set_title(f"{nom} (% de chaque vraie catégorie)")
    ax.set_xlabel("Catégorie prédite")
    ax.set_ylabel("Vraie catégorie")
plt.tight_layout()
plt.show()

Les quatre modèles font les mêmes erreurs. Sports est très bien reconnu (plus de 97 %), son vocabulaire est très particulier. La grosse confusion est entre Business et Sci/Tech : 8,5 % des articles Business sont classés en Sci/Tech par le SVM. C'est ce qu'on avait prévu avec les nuages de mots : les articles sur Apple, Google ou IBM parlent autant d'économie que de technologie.

In [ ]:
meilleur = comparaison.index[0]
print("Meilleur modèle :", meilleur)
print(classification_report(y_test, predictions[meilleur], digits=4))

In [ ]:
masque_erreurs = predictions[meilleur] != y_test
erreurs = test_df.loc[masque_erreurs, ["text", "label_text"]].copy()
erreurs["text"] = erreurs["text"].str[:110]   # texte raccourci pour la lisibilité
erreurs["prediction"] = predictions[meilleur][masque_erreurs]
print(f"{len(erreurs)} erreurs sur {len(test_df)} articles")
erreurs.sample(8, random_state=0)

654 erreurs sur 7 425 articles. En les lisant, on remarque que pas mal d'entre elles sont en fait des erreurs d'étiquette dans le dataset : « Colts Lead Pats Early in Third Quarter » est un match de football américain étiqueté World, et le modèle répond Sports. Pareil pour l'article sur l'académie de cricket. D'autres sont vraiment ambigus, comme le Premier ministre indien qui parle d'inflation (World ou Business ?). Une partie des erreurs est donc impossible à corriger, quel que soit le modèle.

## 5. Deep learning

On teste maintenant deux réseaux de neurones, pour voir s'ils font mieux que les modèles classiques et à quel prix :
- un LSTM bidirectionnel, qui lit le texte mot par mot et tient compte de leur ordre, ce que le TF-IDF ne fait pas ;
- DistilBERT, un modèle de langage déjà entraîné par Google sur énormément de texte anglais, qu'on spécialise sur nos 4 catégories (fine-tuning).

Cette partie a besoin du GPU.

In [ ]:
import os
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU :", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4")

In [ ]:
# Graine aléatoire fixée pour des résultats reproductibles
GRAINE = 42
random.seed(GRAINE)
np.random.seed(GRAINE)
torch.manual_seed(GRAINE);

In [ ]:
# Correspondance numéro ↔ nom de catégorie (0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech)
id2label = train_df.drop_duplicates("label").set_index("label")["label_text"].sort_index().to_dict()
label2id = {v: k for k, v in id2label.items()}
categories_id = list(id2label.values())
id2label

In [ ]:
# Jeu de validation (10 % du train) : pour suivre l'apprentissage et choisir la meilleure époque.
# Le jeu de test n'est utilisé qu'une seule fois, à la fin.
train_part, val_part = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=GRAINE)
train_part, val_part = train_part.reset_index(drop=True), val_part.reset_index(drop=True)
print("Entraînement :", len(train_part), "| Validation :", len(val_part), "| Test :", len(test_df))

On met de côté 10 % du train comme jeu de validation (106 500 articles pour l'entraînement, 11 834 pour la validation). Il sert à suivre l'apprentissage à chaque époque et à garder la meilleure version du modèle. Le test ne sert qu'une fois, à la fin.

In [ ]:
resultats_deep = []
predictions_deep = {}

def taille_dossier_mo(chemin):
    """Taille totale d'un fichier ou d'un dossier, en Mo."""
    chemin = Path(chemin)
    if chemin.is_file():
        return chemin.stat().st_size / 1e6
    return sum(f.stat().st_size for f in chemin.rglob("*") if f.is_file()) / 1e6

def enregistrer_resultat(nom, y_pred, temps_entrainement, taille_mo):
    predictions_deep[nom] = y_pred
    resultats_deep.append({
        "modele": nom,
        "accuracy": accuracy_score(test_df["label"], y_pred),
        "f1_macro": f1_score(test_df["label"], y_pred, average="macro"),
        "temps_entrainement_s": temps_entrainement,
        "taille_mo": taille_mo,
    })
    r = resultats_deep[-1]
    print(f"{nom} : accuracy = {r['accuracy']:.4f}, F1 macro = {r['f1_macro']:.4f}, "
          f"entraînement = {temps_entrainement:.0f} s, taille = {taille_mo:.1f} Mo")

### 5.1 LSTM

Le LSTM lit une suite de mots, pas une matrice TF-IDF. On remplace chaque mot par son numéro dans le vocabulaire (les 30 000 mots les plus fréquents, les autres deviennent « inconnu ») et on coupe ou complète chaque texte à 64 mots.

In [ ]:
# Longueur des textes nettoyés (en mots) pour choisir la longueur des séquences
longueurs = train_part["text_clean"].str.split().str.len()
print(longueurs.describe(percentiles=[0.5, 0.95, 0.99]).round(1))

In [ ]:
MAX_MOTS = 30_000   # taille du vocabulaire
MAX_LEN = 64        # longueur maximale d'une séquence (en mots)

# Vocabulaire construit sur le jeu d'entraînement uniquement
compteur = Counter(mot for texte in train_part["text_clean"] for mot in texte.split())
vocab = {"<pad>": 0, "<unk>": 1}
for mot, _ in compteur.most_common(MAX_MOTS - 2):
    vocab[mot] = len(vocab)

print("Mots distincts dans le train :", len(compteur))
print("Taille du vocabulaire retenu :", len(vocab))

In [ ]:
def encoder(textes):
    """Transforme chaque texte en une suite de numéros de mots, complétée par des 0 jusqu'à MAX_LEN."""
    sequences = np.zeros((len(textes), MAX_LEN), dtype=np.int64)
    longueurs = np.zeros(len(textes), dtype=np.int64)
    for i, texte in enumerate(textes):
        ids = [vocab.get(mot, 1) for mot in texte.split()][:MAX_LEN]
        sequences[i, :len(ids)] = ids
        longueurs[i] = max(len(ids), 1)
    return torch.tensor(sequences), torch.tensor(longueurs)

def chargeur(df, melanger):
    sequences, longueurs = encoder(df["text_clean"])
    donnees = TensorDataset(sequences, longueurs, torch.tensor(df["label"].values))
    return DataLoader(donnees, batch_size=256, shuffle=melanger)

train_loader = chargeur(train_part, melanger=True)
val_loader = chargeur(val_part, melanger=False)
test_loader = chargeur(test_df, melanger=False)

print("Exemple :", train_part["text_clean"][0])
print("Encodé  :", encoder(train_part["text_clean"][:1])[0][0].tolist())

Les textes nettoyés font 24 mots en médiane et 99 % d'entre eux font moins de 43 mots (le plus long en fait 97) : avec 64 mots, on ne coupe quasiment aucun texte. Le train contient 52 406 mots différents, on garde les 30 000 plus fréquents. Ceux qu'on laisse de côté sont des mots très rares, surtout des noms propres et des fautes. Dans l'exemple encodé, les 0 à la fin servent juste à compléter la séquence jusqu'à 64.

In [ ]:
class ClassifieurLSTM(nn.Module):
    def __init__(self, taille_vocab, dim_embedding=128, dim_cachee=128, nb_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, dim_embedding, padding_idx=0)
        self.lstm = nn.LSTM(dim_embedding, dim_cachee, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.sortie = nn.Linear(2 * dim_cachee, nb_classes)

    def forward(self, sequences, longueurs):
        x = self.embedding(sequences)
        # pack_padded_sequence : le LSTM ignore les 0 de remplissage
        x = pack_padded_sequence(x, longueurs.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)   # dernier état caché : sens avant + sens arrière
        return self.sortie(self.dropout(h))

modele_lstm = ClassifieurLSTM(len(vocab)).to(device)
print(modele_lstm)
print("Nombre de paramètres :", sum(p.numel() for p in modele_lstm.parameters()))

Le réseau transforme d'abord chaque mot en un vecteur de 128 nombres (embedding), appris pendant l'entraînement : des mots de sens proche finissent avec des vecteurs proches. Le LSTM lit ensuite la phrase dans les deux sens, et une dernière couche donne un score pour chacune des 4 catégories. Le dropout éteint au hasard 30 % des neurones pendant l'entraînement pour limiter le surapprentissage.

À chaque époque on regarde le F1 sur la validation, on garde la meilleure, et on arrête dès que ça se dégrade.

In [ ]:
def predire_lstm(modele, loader):
    modele.eval()
    sorties = []
    with torch.no_grad():
        for sequences, longueurs, _ in loader:
            sorties.append(modele(sequences.to(device), longueurs).argmax(dim=1).cpu())
    return torch.cat(sorties).numpy()

def entrainer_une_epoque(modele, loader, optimiseur, critere):
    modele.train()
    perte_totale = 0
    for sequences, longueurs, labels in loader:
        optimiseur.zero_grad()
        perte = critere(modele(sequences.to(device), longueurs), labels.to(device))
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item() * len(labels)
    return perte_totale / len(loader.dataset)

In [ ]:
EPOQUES_LSTM = 5
optimiseur = torch.optim.Adam(modele_lstm.parameters(), lr=1e-3)
critere = nn.CrossEntropyLoss()

historique_lstm = []
meilleur_f1, meilleur_etat = 0, None
debut = time.time()

for epoque in range(1, EPOQUES_LSTM + 1):
    perte = entrainer_une_epoque(modele_lstm, train_loader, optimiseur, critere)
    f1_val = f1_score(val_part["label"], predire_lstm(modele_lstm, val_loader), average="macro")
    historique_lstm.append({"epoque": epoque, "perte_train": perte, "f1_validation": f1_val})
    print(f"Époque {epoque} : perte = {perte:.4f}, F1 validation = {f1_val:.4f}")

    if f1_val > meilleur_f1:   # on garde les poids de la meilleure époque
        meilleur_f1 = f1_val
        meilleur_etat = {k: v.clone() for k, v in modele_lstm.state_dict().items()}
    else:                      # arrêt anticipé dès que la validation se dégrade
        print("→ arrêt anticipé")
        break

temps_lstm = time.time() - debut
modele_lstm.load_state_dict(meilleur_etat)
print(f"Temps d'entraînement : {temps_lstm:.0f} s")

In [ ]:
hist = pd.DataFrame(historique_lstm).set_index("epoque")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist["perte_train"].plot(ax=axes[0], marker="o", color="indianred", title="Perte sur l'entraînement")
hist["f1_validation"].plot(ax=axes[1], marker="o", color="steelblue", title="F1 macro sur la validation")
for ax in axes:
    ax.set_xlabel("Époque")
    ax.set_xticks(hist.index)
plt.tight_layout()
plt.show()

In [ ]:
DOSSIER_SORTIE = Path("modeles_deep")
DOSSIER_SORTIE.mkdir(exist_ok=True)

chemin_lstm = DOSSIER_SORTIE / "lstm.pt"
torch.save({"etat": modele_lstm.state_dict(), "vocab": vocab, "max_len": MAX_LEN}, chemin_lstm)

enregistrer_resultat("LSTM", predire_lstm(modele_lstm, test_loader), temps_lstm, taille_dossier_mo(chemin_lstm))

Le réseau a 4,1 millions de paramètres, presque tous dans la couche d'embedding (30 000 mots × 128 nombres).

Pendant l'entraînement, la perte baisse à chaque époque (de 0,54 à 0,14), mais le F1 de validation progresse de moins en moins : il gagne 1 point entre les époques 1 et 2, puis presque plus rien après la troisième (0,900, puis 0,902). Le réseau continue d'apprendre le train de mieux en mieux sans vraiment progresser sur des articles qu'il n'a pas vus, c'est le début du surapprentissage. Le F1 ayant quand même un tout petit peu augmenté à chaque époque, l'arrêt anticipé ne s'est pas déclenché.

Sur le test, le LSTM obtient un F1 de 0,897. C'est moins bien que les trois modèles linéaires sur TF-IDF (de 0,902 à 0,911), alors qu'il met quelques minutes à s'entraîner au lieu de quelques secondes et qu'il pèse 17 Mo. Avec des textes aussi courts (24 mots après nettoyage), l'ordre des mots apporte peu, et un réseau qui part de zéro a besoin de beaucoup de données pour apprendre le sens des mots.

### 5.2 DistilBERT

DistilBERT est une version allégée de BERT. Il connaît déjà l'anglais, on lui ajoute juste une couche de classification et on le réentraîne quelques époques sur nos articles. On lui donne la colonne `text` (sans bruit mais avec les phrases complètes), parce que les mots outils et la ponctuation ont du sens pour lui. Il découpe les mots en sous-mots : un nom inconnu comme DiMarco devient dim, ##ar, ##co, il n'y a donc jamais de mot inconnu.

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)

NOM_MODELE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

# DistilBERT lit la colonne "text" (texte débruité mais NON prétraité) : il a besoin des phrases complètes
exemple = train_part["text"][0]
print(exemple)
print(tokenizer.tokenize(exemple)[:30])

In [ ]:
# Longueur des textes en tokens, pour vérifier que 128 tokens suffisent
nb_tokens = [len(ids) for ids in tokenizer(train_part["text"].sample(min(5000, len(train_part)), random_state=GRAINE).tolist())["input_ids"]]
print(pd.Series(nb_tokens).describe(percentiles=[0.5, 0.95, 0.99]).round(1))
print(f"Textes de plus de 128 tokens : {np.mean(np.array(nb_tokens) > 128) * 100:.2f} %")

In [ ]:
MAX_TOKENS = 128

def vers_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].rename(columns={"label": "labels"}))
    return ds.map(lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_TOKENS),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = vers_dataset(train_part), vers_dataset(val_part), vers_dataset(test_df)
train_ds

Sur un échantillon de 5 000 articles, un texte fait 48 tokens en médiane et 99 % font moins de 93 tokens. Seuls 0,12 % dépassent 128 tokens et seront coupés, donc 128 suffit largement ; une limite plus haute ne ferait que ralentir l'entraînement. On voit aussi sur l'exemple que DistilBERT garde la ponctuation et découpe les mots rares en morceaux (seismologists devient se, ##ism, ##ologists).

In [ ]:
modele_bert = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE, num_labels=len(categories_id), id2label=id2label, label2id=label2id
)

def calculer_metriques(eval_pred):
    logits, labels = eval_pred
    y_pred = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, y_pred), "f1_macro": f1_score(labels, y_pred, average="macro")}

EPOQUES_BERT = 2
arguments = TrainingArguments(
    output_dir="distilbert_checkpoints",
    num_train_epochs=EPOQUES_BERT,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,          # garde la meilleure époque (F1 de validation)
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),       # calcul en demi-précision sur GPU : ~2x plus rapide
    logging_steps=200,
    report_to="none",
    seed=GRAINE,
)

trainer = Trainer(
    model=modele_bert,
    args=arguments,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),   # complète chaque lot à la longueur du plus long texte
    compute_metrics=calculer_metriques,
)

Le message MISSING au chargement est normal : la couche de classification est nouvelle, c'est elle que l'entraînement va apprendre. On fait 2 époques avec un taux d'apprentissage très faible (2e-5) pour ne pas effacer ce que le modèle sait déjà, et on garde la meilleure époque selon le F1 de validation.

In [ ]:
debut = time.time()
trainer.train()
temps_bert = time.time() - debut
print(f"Temps d'entraînement : {temps_bert:.0f} s")

In [ ]:
journal = pd.DataFrame(trainer.state.log_history)
evals = journal.dropna(subset=["eval_f1_macro"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]
evals

In [ ]:
chemin_bert = DOSSIER_SORTIE / "distilbert"
trainer.save_model(chemin_bert)
tokenizer.save_pretrained(chemin_bert)

sortie_test = trainer.predict(test_ds)
enregistrer_resultat("DistilBERT", sortie_test.predictions.argmax(axis=1), temps_bert, taille_dossier_mo(chemin_bert))

C'est de loin l'étape la plus longue du projet, même avec le GPU de Colab (sur notre machine, sans GPU NVIDIA, l'entraînement a pris près d'une heure).

Sur la validation, le F1 est déjà de 0,932 après la première époque et monte à 0,936 après la deuxième. Le modèle est très bon dès le début parce qu'il connaissait déjà l'anglais avant qu'on commence. Sur le test, DistilBERT obtient un F1 de 0,934, soit 2,2 points de plus que le SVM linéaire (0,911) : c'est le meilleur modèle du projet. En contrepartie, le modèle sauvegardé pèse 268 Mo, plus de 36 fois la régression logistique.

In [ ]:
comparaison_deep = pd.DataFrame(resultats_deep).set_index("modele")
comparaison_deep.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, nom in zip(axes, comparaison_deep.index):
    matrice = confusion_matrix(test_df["label"], predictions_deep[nom], normalize="true") * 100
    sns.heatmap(matrice, annot=True, fmt=".1f", cmap="Blues", cbar=False, ax=ax,
                xticklabels=categories_id, yticklabels=categories_id)
    ax.set_title(f"{nom} (% de chaque vraie catégorie)")
    ax.set_xlabel("Catégorie prédite")
    ax.set_ylabel("Vraie catégorie")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(test_df["label"], predictions_deep["DistilBERT"], target_names=categories_id, digits=4))

DistilBERT fait mieux que le LSTM dans toutes les catégories. World passe de 88,5 % à 93,2 % d'articles bien classés, et Sci/Tech de 89,5 % à 92,9 %. Sports reste la catégorie la plus facile (98,6 %).

La confusion entre Business et Sci/Tech reste la principale difficulté : 8,5 % des articles Business sont encore classés en Sci/Tech, exactement comme avec le SVM. DistilBERT corrige surtout le sens inverse (5,3 % des Sci/Tech classés en Business, contre 7,3 % pour le SVM). Même un modèle qui comprend bien l'anglais bute sur les articles qui parlent à la fois d'entreprises et de technologie, sans parler des étiquettes douteuses qu'on a vues plus haut.

## 6. Comparaison de tous les modèles et choix du modèle pour l'API

In [ ]:
colonnes = ["accuracy", "f1_macro", "temps_entrainement_s", "taille_mo"]
tous = pd.concat([comparaison[colonnes], comparaison_deep[colonnes]]).sort_values("f1_macro", ascending=False)
tous.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

couleurs = ["indianred" if m in comparaison_deep.index else "steelblue" for m in tous.index]
axes[0].scatter(tous["taille_mo"], tous["f1_macro"], s=120, c=couleurs)
for nom, ligne in tous.iterrows():
    axes[0].annotate(nom, (ligne["taille_mo"], ligne["f1_macro"]), textcoords="offset points", xytext=(8, 5))
axes[0].axvline(500, color="gray", linestyle="--")
axes[0].text(550, tous["f1_macro"].max(), "limite Vercel\n(500 Mo)", color="gray", ha="left", va="top")
axes[0].set_xscale("log")
axes[0].set_xlim(4, 3000)
axes[0].set_xlabel("Taille du modèle (Mo, échelle log.)")
axes[0].set_ylabel("F1 macro sur le test")
axes[0].set_title("Performance et taille (bleu : classiques, rouge : deep learning)")

tous["temps_entrainement_s"].sort_values().plot.barh(ax=axes[1], color="seagreen", logx=True)
axes[1].set_title("Temps d'entraînement (s, échelle log.)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

DistilBERT est le seul modèle qui se détache vraiment (F1 = 0,934). Derrière, le SVM et la régression logistique sont autour de 0,91 avec seulement 7,4 Mo, et le LSTM fait même un peu moins bien que Naive Bayes.

Sur le graphique de gauche, les modèles les plus intéressants pour une application sont en haut à gauche, c'est-à-dire bons et légers : le SVM et la régression logistique. DistilBERT est tout en haut mais 36 fois plus lourd, et son entraînement n'a rien à voir avec les quelques secondes des modèles linéaires (le graphique de droite est en échelle logarithmique). Le Random Forest cumule les défauts : c'est le moins bon, et à 612 Mo il dépasse à lui seul la limite de Vercel.

Sur le serveur de l'API il n'y aura pas de GPU. On mesure donc le temps que met chaque modèle pour classer un seul article sur un processeur, comme le ferait l'API.

In [ ]:
exemples = test_df["text"].sample(100, random_state=GRAINE).tolist()

# Régression logistique : nettoyage + TF-IDF + prédiction, article par article
debut = time.time()
for texte in exemples:
    modeles["Régression logistique"].predict_proba(vectoriseur.transform([preprocesser(texte)]))
temps_lr = (time.time() - debut) / len(exemples) * 1000

# DistilBERT sur processeur
modele_cpu = trainer.model.to("cpu").eval()
debut = time.time()
with torch.no_grad():
    for texte in exemples:
        modele_cpu(**tokenizer(texte, truncation=True, max_length=MAX_TOKENS, return_tensors="pt"))
temps_bert_cpu = (time.time() - debut) / len(exemples) * 1000

print(f"Régression logistique : {temps_lr:.2f} ms par article")
print(f"DistilBERT (CPU)      : {temps_bert_cpu:.1f} ms par article, soit {temps_bert_cpu / temps_lr:.0f} fois plus lent")

Sur notre machine, la régression logistique classe un article en moins d'une milliseconde, nettoyage compris, et DistilBERT en met environ 9, soit plus de dix fois plus. Pour un utilisateur, 9 ms ne se voient pas : la vitesse n'est donc pas ce qui bloque.

Le vrai problème, c'est le poids. Pour faire tourner DistilBERT, l'API devrait installer PyTorch et transformers, qui font à eux deux environ 750 Mo une fois installés (mesuré sur notre machine), plus les 268 Mo du modèle. On arrive autour d'1 Go, alors que Vercel limite une fonction Python à 500 Mo. Il faudrait passer par une option encore en bêta ou par un serveur payant, et l'API devrait recharger tout ça en mémoire à chaque démarrage.

Le choix se résume donc à 2,2 points de F1 en plus, contre un modèle 36 fois plus lourd, des dépendances qui ne rentrent pas sur Vercel et un entraînement qui se compte en dizaines de minutes au lieu de 3 secondes. Pour une application gratuite que n'importe qui peut tester, le gain ne vaut pas ce coût : on déploie un modèle classique. DistilBERT reste utile dans le projet, il nous dit jusqu'où on pourrait aller.

Reste à choisir entre le SVM et la régression logistique. L'interface affiche la probabilité de chaque catégorie, or le SVM ne donne pas de probabilités. On peut en obtenir en l'entourant d'un `CalibratedClassifierCV`, mais il faut vérifier ce que ça coûte :

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

svm_calibre = CalibratedClassifierCV(LinearSVC(**meilleurs_params["SVM linéaire"]), cv=3).fit(X_train, y_train)
y_pred = svm_calibre.predict(X_test)

print(f"SVM calibré           : F1 macro = {f1_score(y_test, y_pred, average='macro'):.4f}, "
      f"taille = {taille_mo(vectoriseur) + taille_mo(svm_calibre):.1f} Mo")
print(f"Régression logistique : F1 macro = {comparaison.loc['Régression logistique', 'f1_macro']:.4f}, "
      f"taille = {comparaison.loc['Régression logistique', 'taille_mo']:.1f} Mo")

Une fois calibré, le SVM perd son petit avantage : son F1 descend à 0,908, en dessous de la régression logistique (0,909), et il pèse presque deux fois plus lourd (13,8 Mo contre 7,4 Mo), parce que la calibration garde trois SVM, un par pli de la validation croisée.

On retient donc la régression logistique : presque aussi bonne que le meilleur modèle classique, des probabilités sans rien ajouter, 7,4 Mo et 3 secondes d'entraînement.

## 7. Export du modèle et test sur de nouveaux textes

On assemble le vectoriseur TF-IDF et la régression logistique dans un seul objet (`Pipeline`), qu'on sauvegarde avec joblib. C'est ce fichier que l'API charge au démarrage.

In [ ]:
from sklearn.pipeline import make_pipeline

modele_final = make_pipeline(vectoriseur, modeles["Régression logistique"])

Path("models").mkdir(exist_ok=True)
joblib.dump(modele_final, "models/model.joblib")
print(f"models/model.joblib : {Path('models/model.joblib').stat().st_size / 1e6:.1f} Mo")

On le teste comme le fera l'API : le texte passe par `supprimer_bruit`, puis `preprocesser`, puis le modèle. Les phrases ci-dessous sont inventées, elles ne viennent pas du dataset.

In [ ]:
def classer(texte):
    probas = modele_final.predict_proba([preprocesser(supprimer_bruit(texte))])[0]
    return pd.Series(probas, index=modele_final.classes_).sort_values(ascending=False)

phrases = [
    "Real Madrid wins the Champions League final after a late goal from its striker.",
    "The central bank raised interest rates as inflation keeps climbing.",
    "Apple unveils a new iPhone with a faster chip and better battery life.",
    "Thousands of people protest in the capital after the disputed presidential election.",
]
for phrase in phrases:
    resultat = classer(phrase)
    print(f"{resultat.index[0]:<9} ({resultat.iloc[0]:.0%})  {phrase}")

Les quatre phrases sont bien classées, avec une confiance de 91 % à 100 %. La moins sûre est celle sur la banque centrale (91 % en Business), ce qui se comprend : les taux d'intérêt et l'inflation reviennent aussi dans des articles de politique. Le modèle s'en sort sur des sujets bien plus récents que le dataset : l'iPhone n'existait pas en 2004, le modèle se débrouille avec les autres mots de la phrase (apple, chip, battery).

In [ ]:
# Sur Colab : télécharge le modèle (à placer dans le dossier models/ du dépôt)
try:
    from google.colab import files
    files.download("models/model.joblib")
except ImportError:
    pass

## 8. Conclusion


On a construit un classifieur d'articles de presse à partir d'AG News. L'exploration a montré des données bien équilibrées mais bruitées, avec des doublons, des étiquettes contradictoires, une fuite entre train et test et des mentions de source qui auraient permis au modèle de « tricher ». Après nettoyage, il reste 125 759 articles.

| Modèle | F1 macro | Taille |
|---|---|---|
| DistilBERT | 0,934 | 268 Mo |
| SVM linéaire | 0,911 | 7,4 Mo |
| Régression logistique | 0,909 | 7,4 Mo |
| Naive Bayes | 0,902 | 10,6 Mo |
| LSTM | 0,897 | 17 Mo |
| Random Forest | 0,890 | 612 Mo |

DistilBERT est le meilleur, mais il ne gagne que 2,2 points sur les modèles linéaires, pour un modèle 36 fois plus lourd et des dépendances qui ne rentrent pas sur Vercel. On déploie donc la régression logistique sur TF-IDF : F1 de 0,909, 7,4 Mo, une probabilité pour chaque catégorie et une réponse en moins d'une milliseconde.

La principale limite reste la frontière entre Business et Sci/Tech, et une partie des erreurs vient d'étiquettes fausses dans le dataset. Le modèle n'a vu que des articles en anglais de 2004-2005 : il fonctionne sur l'actualité récente tant qu'elle parle des mêmes grands sujets, mais il ne comprend pas d'autres langues. Pour aller plus loin, on pourrait héberger DistilBERT sur un serveur adapté (Hugging Face Spaces par exemple), ou réentraîner le modèle sur des articles plus récents.